In [ ]:
import logging
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger()

FIGURE_DIR = os.path.join("..", "reports", "figures", "heatmap_output")
PARQUET_FILE_PATH = os.path.join("..", "data", "mlmart_base", "ml_mart_base_data.parquet")

os.makedirs(FIGURE_DIR, exist_ok=True)

def plot_heatmap(corr: pd.DataFrame, title: str, filename: str) -> None:
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, linewidths=0.5, linecolor="white", annot_kws={"size": 8}, ax=ax)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    plt.tight_layout()
    
    path = os.path.join(FIGURE_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"  💾 Đã lưu hình ảnh: {path}")

def log_target_corr(corr: pd.DataFrame, target: str = "energy_generated_kwh") -> None:
    if target not in corr.columns:
        return
    result = corr[target].drop(target).sort_values(ascending=False)
    log.info(f"\n── Tương quan với {target} ──")
    log.info("\n" + result.to_string() + "\n")

def main() -> None:
    log.info("=" * 60)
    log.info("  CORRELATION HEATMAP — PARQUET (OPTIMIZED)")
    log.info("=" * 60)

    FEATURES = [
        "energy_generated_kwh", "hour", "day", "month", "year",
        "shortwave_radiation", "direct_normal_irradiance", "diffuse_solar_radiation",
        "temperature_c", "cloud_cover_total", "cloud_cover_low", "cloud_cover_mid",
        "cloud_cover_high", "wind_speed", "precipitation_mm", "sunshine_duration"
    ]

    if os.path.exists(PARQUET_FILE_PATH):
        log.info(f"[Local Data] Tìm thấy file Parquet: {PARQUET_FILE_PATH}")
        df_all = pd.read_parquet(PARQUET_FILE_PATH)
    else:
        log.error(f"LỖI: Không tìm thấy file {PARQUET_FILE_PATH}!")
        return

    query_cols = FEATURES + ["rolling_outlier_flag"]
    available_cols = [col for col in query_cols if col in df_all.columns]
    df_all = df_all[available_cols]

    df_all[FEATURES] = df_all[FEATURES].apply(pd.to_numeric, errors="coerce")
    if df_all["rolling_outlier_flag"].dtype == object:
        df_all["rolling_outlier_flag"] = df_all["rolling_outlier_flag"].astype(str).str.strip().str.lower() == "true"

    log.info("\n[Case 1] Tính toán trên toàn bộ dữ liệu...")
    corr_all = df_all[FEATURES].corr(method="pearson")
    plot_heatmap(corr_all, "Correlation Heatmap – Toàn bộ dữ liệu", "heatmap_01_all_data.png")
    log_target_corr(corr_all)

    log.info("[Case 2] Lọc outlier...")
    df_clean = df_all[df_all["rolling_outlier_flag"] == False]
    corr_clean = df_clean[FEATURES].corr(method="pearson")
    plot_heatmap(corr_clean, "Correlation Heatmap – Đã lọc outlier", "heatmap_02_no_outlier.png")
    log_target_corr(corr_clean)

    log.info("[Case 3] Lọc ban ngày & loại bỏ outlier...")
    df_day = df_clean[
        (df_clean["hour"] >= 6) & 
        (df_clean["hour"] <= 18) & 
        (df_clean["energy_generated_kwh"] > 0)
    ]
    corr_day = df_day[FEATURES].corr(method="pearson")
    plot_heatmap(corr_day, "Correlation Heatmap – Ban ngày, đã lọc outlier", "heatmap_03_daytime_clean.png")
    log_target_corr(corr_day)

    log.info("[Case 4] Tính Spearman correlation...")
    corr_spearman = df_day[FEATURES].corr(method="spearman")
    plot_heatmap(corr_spearman, "Spearman Correlation Heatmap", "heatmap_04_spearman_daytime.png")
    log_target_corr(corr_spearman)

    target = "energy_generated_kwh"
    if target in corr_day.columns and target in corr_spearman.columns:
        compare = pd.DataFrame({
            "Pearson":  corr_day[target].drop(target),
            "Spearman": corr_spearman[target].drop(target),
            "Delta":    (corr_spearman[target] - corr_day[target]).drop(target),
        }).sort_values("Pearson", ascending=False)
        
        csv_compare_path = os.path.join(FIGURE_DIR, "pearson_vs_spearman.csv")
        compare.to_csv(csv_compare_path)
        log.info(f"   Đã lưu CSV phân tích: {csv_compare_path}")

    log.info("=" * 60)
    log.info(f"  Hoàn tất — kết quả lưu tại: {FIGURE_DIR}/")
    log.info("=" * 60)

if __name__ == "__main__":
    main()

21:19:28  ============================================================
21:19:28    CORRELATION HEATMAP — ml_mart.base (OPTIMIZED)
21:19:28  ============================================================
21:19:28  [Cache] Tìm thấy file local: heatmap_output\ml_mart_base_cache.csv. Đang đọc dữ liệu...
21:19:34    Tổng số dòng đọc từ CSV: 2,731,946
21:19:35  
[Case 1] Tính toán trên toàn bộ dữ liệu...
21:19:41    💾 Đã lưu: heatmap_output\heatmap_01_all_data.png
21:19:41  
── Tương quan với energy_generated_kwh ──
21:19:41  
shortwave_radiation         0.532600
direct_normal_irradiance    0.500283
sunshine_duration           0.418871
diffuse_solar_radiation     0.409114
temperature_c               0.282364
hour                        0.073132
wind_speed                  0.065217
year                        0.021218
month                       0.003852
day                         0.003047
cloud_cover_low            -0.027442
cloud_cover_high           -0.034927
precipitation_mm           -0.0